# Experiment 6b: Syncretism + Budget-Matched Dedicated Models

Continuation of Experiment 6 (shared-tree, divergent-linearization). The
big-pool sweep there was clean: a co-training cost ordered by word-order
distance (level 4 > 5 > 6 ≈ 0), fading to ≈0 by d=96. No ε floor, and the
representation-space sharing instruments (overlap, probe transfer, ablation)
were noisy and mutually inconsistent.

Two changes here, addressing the two open problems separately rather than
trying to fix the same instrument again:

1. **ε has nowhere to live in Exp 6's design** (identical tokens/embeddings
   across languages, only child order differs). Fix: **syncretism** — a
   language-dependent surface map that collapses distinct terminals onto the
   same surface token for the non-anchor languages only. This creates a
   genuine embedding-level information loss (the derivation's definition of
   ε) that capacity downstream can only partially undo. Parameterized by
   `EPSILON_LEVEL` (0 = Exp 6's original design, 1 = mild collapse, 2 =
   strong collapse), swept independently of word order so the two axes don't
   get confounded.

2. **Solo comparison wasn't budget-matched** (3 solo models together have
   ~3x a shared model's parameters, so "solo beats shared at low d" partly
   reflects more parameters, not less sharing cost). Fix: **budget-matched
   dedicated models** — for each shared width d, train K=3 dedicated models
   at width d' such that their combined parameter count matches the shared
   model's, instantiating the derivation's actual S-vs-D comparison.

Do NOT run the equal-mix (33/33/33) ablation here — the anchor effect
requires one language to dominate pretraining, so equal mix tests a
different regime and is a later problem.

Word-order design (L_A identity, L_B reverses levels {4,5}, L_C reverses all)
is unchanged from Exp 6 — it already IS head-directionality; relabeling it
SVO/SOV changes the paper's framing, not the computation, and isn't done
here.

**Before training anything**, Section 3 runs a pre-flight check: is the
proposed terminal collapse genuinely ambiguous given local context, or does
a short window already resolve it? This decides whether 4 terminals is
enough or whether `TERM_SIZE` needs to grow.

Primary instruments (Section 8): **terminal-recovery probe** (does the
model's own representation resolve the syncretic ambiguity — the direct ε
readout) and **CE gap vs budget-matched dedicated** (instantiates the
derivation's L_S − L_D). Secondary: **interchange intervention** (causal,
replaces class-mean subspace overlap) and **token-out subspace overlap**
(reused from Exp 6, for consistency checking).


## 0. Device setup

In [7]:
import os
import torch
DEVICE = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  {torch.cuda.get_device_name(2)}")

from contextlib import nullcontext
def autocast_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16) if DEVICE.type == "cuda" else nullcontext()


using device: cuda:2
  NVIDIA H200 NVL


## 1. Config

Unchanged from Exp 6: grammar shape, word-order design, mix, vocab layout.
New: `EPSILON_LEVELS` (the syncretism knob) and `make_surface_map`.

**Design choice (see handoff discussion): L_A never gets a collapse, and
L_B/L_C get the SAME collapse map at a given epsilon level.** The B-vs-C
cost ordering should come from word-order distance (the thing we're already
testing), not from handing one language more syncretism than the other —
conflating the two axes would make the ordering uninterpretable.

In [8]:
# ============================== CONFIG ======================================
LEVELS = 6            # number of NT layers (root=level 1 ... level 6); terminals at level 7
NT_SIZE = 4            # symbols per NT layer, levels 2..6 (root layer has size 1)
TERM_SIZE = 4          # number of terminal symbols
MIN_DEGREE = 2         # min number of production rules per NT symbol
MAX_DEGREE = 4         # max number of production rules per NT symbol
RULE_LEN_CHOICES = [2, 3]   # rule RHS lengths

LANGS = {
    "L_A": set(),                          # identity
    "L_B": {4, 5},                         # partial reversal (irregular)
    "L_C": set(range(1, LEVELS + 1)),      # full reversal (regular, = reverse(L_A))
}
LANG_NAMES = list(LANGS.keys())
MIX = {"L_A": 0.70, "L_B": 0.15, "L_C": 0.15}   # training-data mix, anchor dominates

VOCAB_SIZE = TERM_SIZE + len(LANG_NAMES)
BOS_OFFSET = TERM_SIZE
BOS = {name: BOS_OFFSET + i for i, name in enumerate(LANG_NAMES)}

def round_to_valid_n_embd(x, n_head):
    """Rotary embeddings need head_dim = n_embd/n_head to be EVEN."""
    unit = 2 * n_head
    return max(unit, round(x / unit) * unit)

# --------------------------- SYNCRETISM (new) --------------------------------
# EPSILON_LEVELS[k]: dict[true_terminal] -> collapsed_terminal, applied to the
# SURFACE TOKEN ONLY (vocabulary size and the true terminal identity used for
# grammar/tree-distance bookkeeping are both unaffected). L_A's surface map is
# ALWAYS make_surface_map(0) (identity) regardless of which eps_level is being
# run for B/C -- the anchor never sees its own terminals collapsed.
EPSILON_LEVELS = {
    0: {},                # identity -- Exp 6's original design, control
    1: {3: 2},            # terminal 3 surfaces as token 2 (mild: 1 collapsed pair)
    2: {2: 1, 3: 1},      # terminals 2 and 3 both surface as token 1 (strong)
}

def make_surface_map(eps_level):
    override = EPSILON_LEVELS[eps_level]
    m = {t: t for t in range(TERM_SIZE)}
    m.update(override)
    return m

print("surface maps:")
for k in EPSILON_LEVELS:
    print(f"  eps_level={k}: {make_surface_map(k)}")


surface maps:
  eps_level=0: {0: 0, 1: 1, 2: 2, 3: 3}
  eps_level=1: {0: 0, 1: 1, 2: 2, 3: 2}
  eps_level=2: {0: 0, 1: 1, 2: 1, 3: 1}


## 2. Grammar builder, sampler, linearizer

`linearize` (word order only) and `permute_grammar` (disjoint-control
grammar, unused here but kept for parity with Exp 6) are copied verbatim.
`linearize_surface` is new: applies a per-language surface map to each
terminal AFTER word-order reordering is decided.

`sample_tree_indexed` / `linearize_with_paths` are the *instrumented*
versions from Exp 6 (the ones with instance ids on internal NT nodes) --
needed for the matched eval set and the span-end probes later. Note
`inst` is assigned once, during tree sampling, in canonical order --
it does NOT depend on which language's word order later reads the tree, so
`path[level-1]` (a `(level, idx, inst)` triple) is the SAME key across all
three languages' linearizations of the same underlying tree. That identity
is what lets the interchange intervention (Section 10) match "the same NT
instance" across languages without any extra bookkeeping.

In [9]:
import random
from collections import defaultdict

def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))

    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set()
            tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen:
                    continue
                seen.add(rhs)
                rules[sym].append(rhs)
    return rules, symbols_by_level


def sample_tree(rules, rng, symbol=(1, 0), level=1):
    """Plain (non-indexed) sampler -- used for the pre-flight check and for
    training-data generation, where instance ids aren't needed."""
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(child_sym)
        else:
            children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)


def linearize(node, reverse_levels):
    """True-terminal linearization, word order only (no surface map)."""
    if isinstance(node, int):
        return [node]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order:
        tokens.extend(linearize(c, reverse_levels))
    return tokens


def linearize_surface(node, reverse_levels, surface_map):
    """Like linearize(), but applies surface_map to each terminal AFTER
    order is decided. Vocabulary is unchanged -- surface tokens are a subset
    of the same TERM_SIZE range, so no downstream code needs to know a
    collapse happened unless it explicitly asks (via true_term, added in the
    eps-aware eval set below)."""
    if isinstance(node, int):
        return [surface_map[node]]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order:
        tokens.extend(linearize_surface(c, reverse_levels, surface_map))
    return tokens


def permute_grammar(rules, symbols_by_level, seed):
    """Disjoint-control grammar (unused in this notebook, kept for parity)."""
    rng = random.Random(seed)
    new_rules = {(1, 0): rules[(1, 0)]}
    for level, syms in symbols_by_level.items():
        if level == 1:
            continue
        shuffled = syms[:]
        rng.shuffle(shuffled)
        for original_sym, new_owner in zip(syms, shuffled):
            new_rules[new_owner] = rules[original_sym]
    return new_rules


# --- instrumented sampler/linearizer (from Exp 6, corrected version) ---
def sample_tree_indexed(rules, rng, symbol=(1, 0), level=1, _id_counter=None, _inst_counter=None):
    if _id_counter is None: _id_counter = [0]
    if _inst_counter is None: _inst_counter = [0]
    inst = _inst_counter[0]; _inst_counter[0] += 1
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(("LEAF", child_sym, _id_counter[0])); _id_counter[0] += 1
        else:
            children.append(sample_tree_indexed(rules, rng, child_sym, level + 1, _id_counter, _inst_counter))
    return (symbol, children, inst)


def linearize_with_paths(node, reverse_levels, _path=()):
    """Returns list of (true_terminal, canonical_leaf_id, nt_path) per leaf,
    in this language's linearization order. nt_path[l-1] identifies the
    level-l NT INSTANCE (level, idx, inst) that this leaf descends from --
    `inst` is invariant across languages (assigned once at sampling time)."""
    if isinstance(node, tuple) and len(node) == 3 and node[0] == "LEAF":
        _, tok, leaf_id = node
        return [(tok, leaf_id, _path)]
    (level, idx), children, inst = node
    order = list(reversed(children)) if level in reverse_levels else children
    new_path = _path + ((level, idx, inst),)
    out = []
    for c in order:
        out.extend(linearize_with_paths(c, reverse_levels, new_path))
    return out


def _positions_for_leaves(eval_lang_pack):
    """(n_trees, max_leaf_id+1) LongTensor: (tree, leaf_id) -> sequence pos, or -1."""
    n_trees, seq_len = eval_lang_pack["leaf_id"].shape
    max_lid = int(eval_lang_pack["leaf_id"].max().item()) + 1
    pos = -torch.ones(n_trees, max_lid, dtype=torch.long)
    for t in range(n_trees):
        lids = eval_lang_pack["leaf_id"][t]
        for p in range(seq_len):
            lid = int(lids[p].item())
            if lid >= 0:
                pos[t, lid] = p
    return pos


def spanend_samples(pack, level, tree_ids):
    """(tree, position, label) at the LAST leaf, in THIS language's order, of
    every level-`level` NT instance."""
    pos_of = _positions_for_leaves(pack)
    out = []
    for t in tree_ids:
        best = {}
        for lid, path in pack["path_of"][t].items():
            key = path[level - 1]
            p = int(pos_of[t, lid])
            if key not in best or p > best[key][0]:
                best[key] = (p, key[1])
        out.extend((t, p, lab) for p, lab in best.values())
    return out


def spanend_position_map(pack, level, tree_ids):
    """dict[(tree_id, key)] -> position of the LAST leaf of that NT instance,
    key = (level, idx, inst). Since `inst` is language-invariant, this lets
    two different languages' packs (from the SAME sample_tree_indexed trees)
    be matched by key -- used by the interchange intervention."""
    pos_of = _positions_for_leaves(pack)
    out = {}
    for t in tree_ids:
        best = {}
        for lid, path in pack["path_of"][t].items():
            key = path[level - 1]
            p = int(pos_of[t, lid])
            if key not in best or p > best[key]:
                best[key] = p
        for key, p in best.items():
            out[(t, key)] = p
    return out


# smoke test
rules, symbols_by_level = build_grammar(seed=0)
tree = sample_tree(rules, random.Random(1))
a = linearize(tree, LANGS["L_A"])
c = linearize(tree, LANGS["L_C"])
print(f"sample tree yields {len(a)} terminal tokens under L_A")
print("L_C == reverse(L_A)? ", c == list(reversed(a)))
a_surf1 = linearize_surface(tree, LANGS["L_A"], make_surface_map(0))
b_surf1 = linearize_surface(tree, LANGS["L_B"], make_surface_map(1))
print(f"L_A surface (eps=0) == L_A true terminals? {a_surf1 == a}  (should be True -- anchor never collapses)")
print(f"L_B surface (eps=1) first 10: {b_surf1[:10]}")


sample tree yields 225 terminal tokens under L_A
L_C == reverse(L_A)?  True
L_A surface (eps=0) == L_A true terminals? True  (should be True -- anchor never collapses)
L_B surface (eps=1) first 10: [2, 2, 0, 2, 1, 2, 1, 1, 0, 1]


## 3. Model — TinyRotaryGPT (unchanged from Exp 6)

In [14]:
import torch.nn as nn
import torch.nn.functional as F

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert (n_embd // n_head) % 2 == 0, "head_dim must be EVEN for rotary embeddings"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def forward_with_patch(self, idx, patch_layer=None, patch_mask=None, patch_values=None):
        """Manual forward that can overwrite the residual stream right after
        block `patch_layer`, at positions where patch_mask is True, with
        patch_values. Returns hidden states PRE ln_f/head (same convention as
        extract_hidden_states' forward hooks). Used by the interchange
        intervention (Section 10) -- not used during ordinary training/eval."""
        x = self.tok_emb(idx)
        for l, blk in enumerate(self.blocks):
            x = blk(x)
            if patch_layer is not None and l == patch_layer:
                x = x.clone()
                x[patch_mask] = patch_values.to(x.dtype)
        return x

# smoke test
torch.manual_seed(0)
_m = TinyRotaryGPT(VOCAB_SIZE, n_embd=8, n_layer=2, n_head=2, max_seq_len=64).to(DEVICE)
_idx = torch.randint(0, VOCAB_SIZE, (4, 32), device=DEVICE)
_logits, _ = _m(_idx, _idx)
print(f"forward pass OK on {DEVICE}, output shape:", _logits.shape)


forward pass OK on cuda:2, output shape: torch.Size([4, 32, 7])


## 4. Grammar instance + global run constants

Same grammar seed as Exp 6's big pool (`build_grammar(seed=0)`), so results
are directly comparable. `N_TREES_CHECK`, `SEQ_LEN_CHECK`, `FIXED_BUDGET_SWEEP`,
`N_HEAD` match the big-pool sweep exactly.

In [10]:
check_rules, check_symbols_by_level = build_grammar(seed=0)

N_TREES_CHECK = 50_000
SEQ_LEN_CHECK = 256
FIXED_BUDGET_SWEEP = 90_000
N_HEAD = 4
SEEDS3 = [0, 7, 13]
D_LIST_EXP6 = [8, 16, 24, 32, 48, 96]      # Exp 6's original sweep, for reference
MAX_PAR = 20                                 # parallel training workers; raise if nvidia-smi shows headroom

for d in D_LIST_EXP6:
    assert d % N_HEAD == 0 and (d // N_HEAD) % 2 == 0, f"d={d}: head_dim must be even for rotary"

print(f"grammar built (seed=0): {sum(len(v) for v in check_rules.values())} total rules across "
      f"{len(check_rules)} symbols")


grammar built (seed=0): 65 total rules across 21 symbols


## 5. Pre-flight ambiguity check (NO training)

Before spending any compute: is the proposed terminal collapse genuinely
ambiguous, or does a short local window already resolve it? For each window
size K, hash the K surface tokens immediately preceding every occurrence of
an ambiguous surface token (in that language's own causal order), and check
whether the TRUE terminal is uniquely determined by that local history
alone. Trees have ~238 leaves; if ambiguity has fully resolved by K=32, the
collapse is a weak, locally-memorizable pattern -- not a real test of ε.
If it persists at generous K, that's evidence of genuine, context-hungry
information loss.

Note this measures INFORMATION AVAILABILITY (an upper bound on what any
model, however large, could recover from that much left-context), not what
a trained model actually does with it -- that's Section 8's job.

In [11]:
def preflight_ambiguity_check(rules, eps_level, lang="L_B", n_trees=20_000,
                              window_sizes=(4, 8, 16, 32, 64), seed=0):
    surf = make_surface_map(eps_level)
    ambiguous_surfaces = {}
    for true_t, s in surf.items():
        ambiguous_surfaces.setdefault(s, set()).add(true_t)
    ambiguous_surfaces = {s for s, ts in ambiguous_surfaces.items() if len(ts) > 1}
    if not ambiguous_surfaces:
        print(f"eps_level={eps_level}: no ambiguous surface tokens (identity map)")
        return {}

    rng = random.Random(seed)
    revset = LANGS[lang]
    ctx_to_true = {K: {} for K in window_sizes}
    n_ambiguous_positions = 0

    def walk(node, out):
        if isinstance(node, int):
            out.append(node); return
        (level, idx), children = node
        order = list(reversed(children)) if level in revset else children
        for c in order:
            walk(c, out)

    for _ in range(n_trees):
        tree = sample_tree(rules, rng)
        toks_true = []
        walk(tree, toks_true)
        surf_stream = [surf[t] for t in toks_true]
        for p, (true_t, s) in enumerate(zip(toks_true, surf_stream)):
            if s not in ambiguous_surfaces:
                continue
            n_ambiguous_positions += 1
            for K in window_sizes:
                ctx = tuple(surf_stream[max(0, p - K):p])
                ctx_to_true[K].setdefault(ctx, set()).add(true_t)

    print(f"\n[{lang}, eps_level={eps_level}] pre-flight ambiguity check over {n_trees} trees "
          f"({n_ambiguous_positions} ambiguous-surface occurrences)")
    print(f"{'window K':>9s} {'unique ctx':>11s} {'still ambiguous':>16s} {'ambiguous frac':>15s}")
    out = {}
    for K in window_sizes:
        d = ctx_to_true[K]
        n_ambig_ctx = sum(1 for v in d.values() if len(v) > 1)
        frac = n_ambig_ctx / max(1, len(d))
        out[K] = dict(n_contexts=len(d), n_ambiguous_contexts=n_ambig_ctx, frac_ambiguous=frac)
        print(f"{K:>9d} {len(d):>11d} {n_ambig_ctx:>16d} {frac:>15.3f}")
    return out


print("=" * 78)
print("If 'still ambiguous' stays > 0 even at K=64 (~1/4 of a tree's leaves),")
print("the collapse is a genuine information loss, not a locally-memorizable")
print("pattern a shortcut could learn. If it hits 0 by K=8-16, the collapse is")
print("too weak at TERM_SIZE=4 -- consider raising TERM_SIZE before training.")
print("=" * 78)
preflight_1_B = preflight_ambiguity_check(check_rules, eps_level=1, lang="L_B")
preflight_1_C = preflight_ambiguity_check(check_rules, eps_level=1, lang="L_C")
preflight_2_B = preflight_ambiguity_check(check_rules, eps_level=2, lang="L_B")
preflight_2_C = preflight_ambiguity_check(check_rules, eps_level=2, lang="L_C")


If 'still ambiguous' stays > 0 even at K=64 (~1/4 of a tree's leaves),
the collapse is a genuine information loss, not a locally-memorizable
pattern a shortcut could learn. If it hits 0 by K=8-16, the collapse is
too weak at TERM_SIZE=4 -- consider raising TERM_SIZE before training.

[L_B, eps_level=1] pre-flight ambiguity check over 20000 trees (1691984 ambiguous-surface occurrences)
 window K  unique ctx  still ambiguous  ambiguous frac
        4          79               60           0.759
        8        2883             1737           0.602
       16      582236           120035           0.206
       32     1601668             3071           0.002
       64     1601767             3044           0.002

[L_C, eps_level=1] pre-flight ambiguity check over 20000 trees (1691984 ambiguous-surface occurrences)
 window K  unique ctx  still ambiguous  ambiguous frac
        4          86               60           0.698
        8        2994             1819           0.608
       16    

## 6. Data generation with syncretism

`make_shared_tree_mix_eps` / `make_monolingual_dataset_eps`: same structure
as Exp 6's data functions, but linearize through `linearize_surface` with a
per-language surface map at the given `eps_level`. L_A is always
`make_surface_map(0)` regardless of what eps_level is being tested for B/C.

`build_matched_eval_set_eps`: extends Exp 6's matched eval set with a
`true_term` tensor alongside the usual `idx` (surface tokens the model
actually sees). This is what makes the terminal-recovery probe (Section 8)
possible -- we can ask "does the hidden state at this position recover the
TRUE terminal" even though the model only ever saw the (possibly collapsed)
surface token.

In [12]:
def make_shared_tree_mix_eps(rules, n_trees, seed, seq_len, eps_level, mix=MIX):
    surf = {"L_A": make_surface_map(0), "L_B": make_surface_map(eps_level), "L_C": make_surface_map(eps_level)}
    rng = random.Random(seed)
    stream = []
    for _ in range(n_trees):
        lang = rng.choices(LANG_NAMES, weights=[mix[n] for n in LANG_NAMES])[0]
        tree = sample_tree(rules, rng)
        stream.append(BOS[lang])
        stream.extend(linearize_surface(tree, LANGS[lang], surf[lang]))
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (seq_len + 1)
    return stream[: n_chunks * (seq_len + 1)].view(n_chunks, seq_len + 1)


def make_monolingual_dataset_eps(rules, lang_name, n_trees, seed, seq_len, eps_level):
    """For budget-matched dedicated models. Anchor (L_A) should only ever be
    called with eps_level=0 -- it never gets a collapse."""
    surf = make_surface_map(0 if lang_name == "L_A" else eps_level)
    rng = random.Random(seed)
    revset = LANGS[lang_name]
    stream = []
    for _ in range(n_trees):
        tree = sample_tree(rules, rng)
        stream.append(BOS[lang_name])
        stream.extend(linearize_surface(tree, revset, surf))
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (seq_len + 1)
    return stream[: n_chunks * (seq_len + 1)].view(n_chunks, seq_len + 1)


def build_matched_eval_set_eps(rules, n_trees, seed, seq_len, eps_level, max_leaves=None):
    """Same trees across all three langs (matched via canonical leaf_id), now
    also carrying true_term (the TRUE terminal, always visible regardless of
    surface collapse) alongside idx (the surface token, what the model saw)."""
    surf = {"L_A": make_surface_map(0), "L_B": make_surface_map(eps_level), "L_C": make_surface_map(eps_level)}
    rng = random.Random(seed)
    cap = seq_len - 1
    trees_kept = []
    while len(trees_kept) < n_trees:
        tree = sample_tree_indexed(rules, rng)
        n_leaves = len(linearize_with_paths(tree, LANGS["L_A"]))
        if n_leaves > cap:
            continue
        if max_leaves is not None and n_leaves > max_leaves:
            continue
        trees_kept.append(tree)

    out = {}
    for lang in LANG_NAMES:
        idx       = torch.zeros(n_trees, seq_len, dtype=torch.long)
        true_term = -torch.ones(n_trees, seq_len, dtype=torch.long)
        mask      = torch.zeros(n_trees, seq_len, dtype=torch.bool)
        leaf_id   = -torch.ones(n_trees, seq_len, dtype=torch.long)
        path_of   = []
        for t, tree in enumerate(trees_kept):
            linearized = linearize_with_paths(tree, LANGS[lang])
            idx[t, 0] = BOS[lang]
            per_tree_paths = {}
            for pos, (tok, lid, pth) in enumerate(linearized):
                idx[t, pos + 1] = surf[lang][tok]
                true_term[t, pos + 1] = tok
                mask[t, pos + 1] = True
                leaf_id[t, pos + 1] = lid
                per_tree_paths[lid] = pth
            path_of.append(per_tree_paths)
        out[lang] = dict(idx=idx, true_term=true_term, mask=mask, leaf_id=leaf_id, path_of=path_of)
    return out


# smoke test
_es0 = build_matched_eval_set_eps(check_rules, 20, seed=1, seq_len=SEQ_LEN_CHECK, eps_level=1)
_n_collapsed_positions = int((_es0["L_B"]["idx"] != _es0["L_B"]["true_term"]).sum())
print(f"eps_level=1, L_B: {_n_collapsed_positions} positions where surface token != true terminal "
      f"(out of {int(_es0['L_B']['mask'].sum())} leaf positions)")
assert (_es0["L_A"]["idx"] == _es0["L_A"]["true_term"])[_es0["L_A"]["mask"]].all(), "L_A must never collapse"
print("L_A surface == true terminal everywhere: OK (anchor never collapses)")


eps_level=1, L_B: 1529 positions where surface token != true terminal (out of 4053 leaf positions)
L_A surface == true terminal everywhere: OK (anchor never collapses)


## 7. Budget-matched dedicated width calculator

For the derivation's actual comparison (shared circuit with budget N vs K
dedicated circuits splitting N), find, for each shared width `d`, the
dedicated width `d'` such that `K * params(d') ≈ params(d)`. Computed
empirically (instantiate the model, count parameters) rather than derived
by hand -- parameter count is not exactly quadratic in d (linear terms from
biases/LayerNorms), so a closed-form d/sqrt(K) is only approximate.

In [15]:
def count_params(d, n_layer=12, n_head=N_HEAD, seq_len=SEQ_LEN_CHECK, vocab_size=VOCAB_SIZE):
    m = TinyRotaryGPT(vocab_size, n_embd=d, n_layer=n_layer, n_head=n_head, max_seq_len=seq_len + 1)
    return sum(p.numel() for p in m.parameters())


def find_budget_matched_width(d_shared, K=3, n_head=N_HEAD, candidates=None):
    """Smallest-error d' (rotary-valid) such that K * params(d') ~= params(d_shared)."""
    target = count_params(d_shared, n_head=n_head)
    if candidates is None:
        unit = 2 * n_head
        candidates = list(range(unit, d_shared + 1, unit))
    best = min(candidates, key=lambda dp: abs(K * count_params(dp, n_head=n_head) - target))
    ratio = K * count_params(best, n_head=n_head) / target
    return best, ratio


BM_SHARED_DS = [16, 32, 96]     # shared widths to build a budget-matched comparison for
BM_WIDTHS = {}
print(f"{'d_shared':>9s} {'params(d)':>12s} {'d_prime':>8s} {'K*params(d_prime)/params(d)':>28s}")
for d in BM_SHARED_DS:
    dp, ratio = find_budget_matched_width(d, K=3, n_head=N_HEAD)
    BM_WIDTHS[d] = dp
    print(f"{d:>9d} {count_params(d):>12,d} {dp:>8d} {ratio:>28.3f}")


 d_shared    params(d)  d_prime  K*params(d_prime)/params(d)
       16       39,616        8                        0.802
       32      152,960       16                        0.777
       96    1,343,616       56                        1.030


## 8. Training infrastructure

`train_worker.py` is copied VERBATIM from Exp 6 -- the training loop never
changes across conditions; only the DATA differs (syncretism is baked into
the saved tensor at generation time, budget matching only changes which
width `--d` a worker trains at). `run_parallel_jobs` generalizes Exp 6's
launcher to take an arbitrary job list, so all three training conditions
below reuse the same code path.

In [16]:
%%writefile train_worker.py
import argparse, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert (n_embd // n_head) % 2 == 0, "head_dim must be EVEN for rotary embeddings"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--d", type=int, required=True)
    ap.add_argument("--seed", type=int, required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--vocab", type=int, required=True)
    ap.add_argument("--seq_len", type=int, required=True)
    ap.add_argument("--steps", type=int, default=90000)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_layer", type=int, default=12)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--compile", type=int, default=0)
    a = ap.parse_args()

    torch.set_num_threads(1)
    device = torch.device(a.device)
    torch.manual_seed(a.seed)
    data = torch.load(a.data).to(device)
    model = TinyRotaryGPT(a.vocab, n_embd=a.d, n_layer=a.n_layer, n_head=a.n_head,
                          max_seq_len=a.seq_len + 1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, fused=True)
    fwd = torch.compile(model) if a.compile else model
    n = data.shape[0]
    buf = torch.zeros(a.steps, device=device)
    t0 = time.time()
    for step in range(a.steps):
        idx = torch.randint(0, n, (a.batch,), device=device)
        batch = data[idx]
        x, y = batch[:, :-1], batch[:, 1:]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = fwd(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        buf[step] = loss.detach().float()
        if (step + 1) % 5000 == 0:
            print(f"step {step+1:>7d}: mean loss over last 5000 = {buf[step-4999:step+1].mean().item():.4f}"
                  f"  ({(step + 1) / (time.time() - t0):.0f} it/s)", flush=True)
        if (step + 1) % 10000 == 0 or step == a.steps - 1:
            tmp = a.ckpt + ".tmp"
            torch.save({"model_state": model.state_dict(), "opt_state": opt.state_dict(),
                        "step": step + 1, "losses": buf[:step + 1].tolist()}, tmp)
            os.replace(tmp, a.ckpt)

if __name__ == "__main__":
    main()


Overwriting train_worker.py


In [17]:
import subprocess, sys, time

def ckpt_step(path):
    """Step count stored in a checkpoint, or None if the file doesn't exist."""
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location="cpu", weights_only=False)["step"]


def run_parallel_jobs(jobs, vocab_size, seq_len, max_par=MAX_PAR, device=DEVICE):
    """jobs: list of dicts with keys d, seed, data_path, ckpt_path, n_head, steps.
    Skips jobs already complete at the target step count. Reuses
    train_worker.py unchanged -- the training loop is identical across every
    condition in this notebook; only the DATA (saved as a .pt tensor) differs."""
    jobs = [j for j in jobs if ckpt_step(j["ckpt_path"]) != j["steps"]]
    print(f"{len(jobs)} runs to do, {max_par} at a time")

    def launch(j):
        log = open(j["ckpt_path"] + ".log", "w")
        cmd = [sys.executable, "train_worker.py", "--d", str(j["d"]), "--seed", str(j["seed"]),
               "--data", j["data_path"], "--ckpt", j["ckpt_path"], "--vocab", str(vocab_size),
               "--seq_len", str(seq_len), "--steps", str(j["steps"]), "--n_head", str(j["n_head"]),
               "--device", str(device)]
        return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT), j, log

    running = []
    while jobs or running:
        still = []
        for p, j, log in running:
            rc = p.poll()
            if rc is None:
                still.append((p, j, log))
            else:
                log.close()
                tag = f"d={j['d']} seed={j['seed']} ({os.path.basename(j['ckpt_path'])})"
                print(f"finished {tag}  exit code {rc}" + ("" if rc == 0 else "   <-- check the .log file"))
        running = still
        while jobs and len(running) < max_par:
            running.append(launch(jobs.pop(0)))
        time.sleep(10)
    print("all queued jobs finished (check exit codes above for failures)")


def assert_all_complete(paths_and_steps):
    missing = [(p, s, ckpt_step(p)) for p, s in paths_and_steps if ckpt_step(p) != s]
    assert not missing, f"incomplete runs (path, target_step, actual_step): {missing}"
    print(f"all {len(paths_and_steps)} runs complete")


## 9. Training — Condition A: syncretism shared sweep

Shared model trained on the eps-mixed tree data, at `eps_level` in {1, 2},
`d` in {8, 16, 32, 96} (dropping 24, 48 relative to Exp 6's full sweep --
these four points bracket the crossover region and the endpoints), 3 seeds.
`eps_level=0` is Exp 6's original big-pool sweep and is NOT retrained here
(reuse the existing checkpoints if available; otherwise adjust `EPS_D_LIST`
to include 0 and this cell reproduces it identically).

In [18]:
EPS_D_LIST = [8, 16, 32, 96]
CK_EPS = "./exp6b_syncretism"; os.makedirs(CK_EPS, exist_ok=True)

def eps_shared_path(eps_level, d, seed):
    return os.path.join(CK_EPS, f"shared_eps{eps_level}_d{d}_seed{seed}.pt")

def eps_data_path(eps_level, seed):
    return os.path.join(CK_EPS, f"data_eps{eps_level}_seed{seed}.pt")

# 1) precompute + save one dataset per (eps_level, seed), reused across all d
for eps_level in (1, 2):
    for seed in SEEDS3:
        data_seed = seed if seed != 0 else 1
        p = eps_data_path(eps_level, data_seed)
        if not os.path.exists(p):
            data = make_shared_tree_mix_eps(check_rules, n_trees=N_TREES_CHECK, seed=data_seed,
                                            seq_len=SEQ_LEN_CHECK, eps_level=eps_level, mix=MIX)
            torch.save(data.cpu(), p)
            print(f"saved {p}")

# 2) build and launch the job list
eps_jobs = []
for eps_level in (1, 2):
    for d in EPS_D_LIST:
        for seed in SEEDS3:
            data_seed = seed if seed != 0 else 1
            eps_jobs.append(dict(d=d, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                                 data_path=eps_data_path(eps_level, data_seed),
                                 ckpt_path=eps_shared_path(eps_level, d, seed)))

run_parallel_jobs(eps_jobs, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in eps_jobs])


0 runs to do, 20 at a time
all queued jobs finished (check exit codes above for failures)
all 24 runs complete


## 10. Training — Condition B: budget-matched dedicated models

For each `d_shared` in `BM_SHARED_DS`, train K=3 dedicated (monolingual)
models at the matched width `d'` from Section 7, for languages B and C
(A is the anchor and never separates in the theory -- no dedicated A model
is needed for the comparison). Run at `eps_level` in {0, 1}: eps=0 tests
whether sharing beats budget-matched dedicated at low d even with NO ε
(pure capacity-pooling benefit); eps=1 tests whether the crossover shifts.

Each dedicated model gets the FULL step budget (90K), not a fraction of
it -- the derivation splits PARAMETERS across the K dedicated circuits, not
training compute.

In [19]:
CK_BM = "./exp6b_budget_matched"; os.makedirs(CK_BM, exist_ok=True)

def bm_ckpt_path(eps_level, lang, dprime, seed):
    return os.path.join(CK_BM, f"solo_{lang}_eps{eps_level}_d{dprime}_seed{seed}.pt")

def bm_data_path(eps_level, lang, seed):
    return os.path.join(CK_BM, f"data_{lang}_eps{eps_level}_seed{seed}.pt")

# 1) precompute + save one monolingual dataset per (lang, eps_level, seed)
for eps_level in (0, 1):
    for lang in ("L_B", "L_C"):
        for seed in SEEDS3:
            data_seed = seed if seed != 0 else 1
            p = bm_data_path(eps_level, lang, data_seed)
            if not os.path.exists(p):
                data = make_monolingual_dataset_eps(check_rules, lang, n_trees=N_TREES_CHECK,
                                                    seed=data_seed, seq_len=SEQ_LEN_CHECK, eps_level=eps_level)
                torch.save(data.cpu(), p)
                print(f"saved {p}")

# 2) build and launch the job list -- dedupe by d' (several d_shared can map
#    to the same matched width, no need to train it twice)
bm_jobs = []
seen_dprime = set()
for d in BM_SHARED_DS:
    dprime = BM_WIDTHS[d]
    if dprime in seen_dprime:
        continue
    seen_dprime.add(dprime)
    for eps_level in (0, 1):
        for lang in ("L_B", "L_C"):
            for seed in SEEDS3:
                data_seed = seed if seed != 0 else 1
                bm_jobs.append(dict(d=dprime, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                                    data_path=bm_data_path(eps_level, lang, data_seed),
                                    ckpt_path=bm_ckpt_path(eps_level, lang, dprime, seed)))

run_parallel_jobs(bm_jobs, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in bm_jobs])


0 runs to do, 20 at a time
all queued jobs finished (check exit codes above for failures)
all 36 runs complete


## 11. Training — Condition C: A′ calibration

A fourth language, `L_A_PRIME`: same word order as A (identity) and same
surface map as A (no collapse), different BOS tag. Mix becomes
35/35/15/15 (A/A′/B/C). This is the calibration ceiling for every
representation-level instrument in Section 8 -- A and A′ compute IDENTICALLY,
so any instrument applied to the A→A′ pair should report "fully shared."
If it doesn't, the instrument (not the language) is the problem.

Kept in its own namespace (`LANGS_A`, `BOS_A`, `VOCAB_A`) rather than
mutating the global `LANGS`/`BOS`/`VOCAB_SIZE` used everywhere else in this
notebook -- avoids exactly the shared-global-state bug class flagged in Exp
5 (the unseeded-RNG incident). Only run at `eps_level=1` (calibrating the
syncretism condition specifically).

In [20]:
LANGS_A = dict(LANGS); LANGS_A["L_A_PRIME"] = set()          # identity order, same as L_A
LANG_NAMES_A = LANG_NAMES + ["L_A_PRIME"]
MIX_A = {"L_A": 0.35, "L_A_PRIME": 0.35, "L_B": 0.15, "L_C": 0.15}
VOCAB_A = TERM_SIZE + len(LANG_NAMES_A)
BOS_A = {name: TERM_SIZE + i for i, name in enumerate(LANG_NAMES_A)}

def make_shared_tree_mix_aprime(rules, n_trees, seed, seq_len, eps_level, mix=MIX_A):
    surf = {"L_A": make_surface_map(0), "L_A_PRIME": make_surface_map(0),
            "L_B": make_surface_map(eps_level), "L_C": make_surface_map(eps_level)}
    rng = random.Random(seed)
    stream = []
    for _ in range(n_trees):
        lang = rng.choices(LANG_NAMES_A, weights=[mix[n] for n in LANG_NAMES_A])[0]
        tree = sample_tree(rules, rng)
        stream.append(BOS_A[lang])
        stream.extend(linearize_surface(tree, LANGS_A[lang], surf[lang]))
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (seq_len + 1)
    return stream[: n_chunks * (seq_len + 1)].view(n_chunks, seq_len + 1)


APRIME_D_LIST = [16, 32, 96]
CK_APRIME = "./exp6b_aprime"; os.makedirs(CK_APRIME, exist_ok=True)

def aprime_path(d, seed):
    return os.path.join(CK_APRIME, f"shared_aprime_eps1_d{d}_seed{seed}.pt")

def aprime_data_path(seed):
    return os.path.join(CK_APRIME, f"data_aprime_eps1_seed{seed}.pt")

for seed in SEEDS3:
    data_seed = seed if seed != 0 else 1
    p = aprime_data_path(data_seed)
    if not os.path.exists(p):
        data = make_shared_tree_mix_aprime(check_rules, n_trees=N_TREES_CHECK, seed=data_seed,
                                           seq_len=SEQ_LEN_CHECK, eps_level=1, mix=MIX_A)
        torch.save(data.cpu(), p)
        print(f"saved {p}")

aprime_jobs = []
for d in APRIME_D_LIST:
    for seed in SEEDS3:
        data_seed = seed if seed != 0 else 1
        aprime_jobs.append(dict(d=d, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                                data_path=aprime_data_path(data_seed), ckpt_path=aprime_path(d, seed)))

run_parallel_jobs(aprime_jobs, vocab_size=VOCAB_A, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in aprime_jobs])


0 runs to do, 20 at a time
all queued jobs finished (check exit codes above for failures)
all 9 runs complete


## 12. Evaluation infrastructure (reused from Exp 6)

Copied essentially verbatim: `load_shared_ckpt`, `extract_hidden_states`,
`split_trees`, `mean_ce`, `per_language_loss`, `_fit_logreg`/`_score`,
`_spanend_hidden`, `_disc_subspace`, `_overlap`. `load_shared_ckpt` needs a
`vocab_size` argument now (was implicitly `VOCAB_SIZE` in Exp 6) since A′
checkpoints use a different vocab than everything else in this notebook.

In [21]:
def load_shared_ckpt(path, d_model, seq_len, vocab_size=VOCAB_SIZE, n_layer=12, n_head=N_HEAD, device=DEVICE):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m = TinyRotaryGPT(vocab_size, n_embd=d_model, n_layer=n_layer, n_head=n_head,
                      max_seq_len=seq_len + 1).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    tail = ckpt["losses"][-1000:]
    print(f"{path}: step={ckpt['step']}, tail-1000 mean loss={sum(tail)/len(tail):.4f}")
    return m


@torch.no_grad()
def extract_hidden_states(model, idx_tensor, layers, batch_size=32, device=DEVICE):
    model.eval()
    activations = {l: [] for l in layers}

    def make_hook(l):
        def hook(module, inp, out):
            activations[l].append(out.detach().float().cpu())
        return hook

    handles = [model.blocks[l].register_forward_hook(make_hook(l)) for l in layers]
    try:
        for i in range(0, idx_tensor.shape[0], batch_size):
            batch = idx_tensor[i:i + batch_size].to(device)
            with autocast_ctx():
                _ = model(batch)
    finally:
        for h in handles:
            h.remove()
    return {l: torch.cat(activations[l], dim=0) for l in layers}


def split_trees(n_trees, test_frac=0.2, seed=0):
    import numpy as np
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n_trees)
    n_test = int(n_trees * test_frac)
    return set(idx[n_test:].tolist()), set(idx[:n_test].tolist())


@torch.no_grad()
def mean_ce(model, data, bs=256):
    model.eval(); tot = n = 0
    for i in range(0, data.shape[0], bs):
        b = data[i:i + bs].to(DEVICE)
        x, y = b[:, :-1], b[:, 1:]
        with autocast_ctx():
            _, loss = model(x, y)
        tot += loss.item() * x.numel(); n += x.numel()
    return tot / n


def per_language_loss(model, data, bos_token, device=DEVICE):
    """Like Exp 6's per_language_loss, but takes the BOS token id directly
    (works for both the 3-language and 4-language/A' vocabularies). NOTE:
    this scores the WHOLE row after the first BOS -- fine for solo/monolingual
    data (one language per row), but on a MIXED shared-tree row this can
    silently average in a DIFFERENT language's tree if one is packed later in
    the same row. Use first_tree_ce below for shared (mixed-row) data."""
    mask = data[:, 0] == bos_token
    subset = data[mask]
    if subset.shape[0] == 0:
        return None
    x, y = subset[:, :-1].to(device), subset[:, 1:].to(device)
    with torch.no_grad(), autocast_ctx():
        _, loss = model(x, y)
    return loss.item()


@torch.no_grad()
def first_tree_ce(model, data, bos_token, all_bos_ids, bs=256, device=DEVICE):
    """CE restricted to the FIRST tree in each row: masks any position at or
    after a SECOND BOS tag appears in that row. Exp 6's shared-tree mix packs
    trees end-to-end into fixed-length rows, so a row starting with
    `bos_token` can still contain a DIFFERENT language's tree later in the
    same row -- per_language_loss above would silently average that in. This
    is the exact correction Exp 6's handoff used for its headline shared-CE
    numbers ('first tree only, tag and everything after masked'); use this,
    not per_language_loss, for CE_shared on mixed-row data."""
    rows = data[data[:, 0] == bos_token]
    if rows.shape[0] == 0:
        return None
    tot = n = 0
    for i in range(0, rows.shape[0], bs):
        b = rows[i:i + bs].to(device)
        x, y = b[:, :-1], b[:, 1:]
        with autocast_ctx():
            logits, _ = model(x)
        ce = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),
                             y.reshape(-1), reduction="none").view(y.shape)
        in_first = (torch.isin(y, all_bos_ids.to(device)).cumsum(1) == 0)
        tot += (ce * in_first).sum().item(); n += in_first.sum().item()
    return tot / n


def cc_safe(o, other, anchor="L_A"):
    """Chance-corrected overlap of the anchor with `other` ('L_B' or 'L_C'),
    from a run_subspace_overlap() result cell; nan if the split-half floor is
    barely above the null (overlap expected by chance alone). Reused from
    Exp 6 -- dropped from the first draft of this notebook by oversight."""
    key = "ab" if other == "L_B" else "ac"
    denom = min(o["floor"][anchor], o["floor"][other]) - o["null"]
    return (o[key] - o["null"]) / denom if denom > 0.1 else float("nan")


def _fit_logreg(X, y, n_class, epochs=300, lr=1e-2, device=DEVICE):
    mu, sd = X.mean(0), X.std(0) + 1e-6
    Xs = ((X - mu) / sd).to(device); ys = y.to(device)
    W = nn.Linear(X.shape[1], n_class).to(device)
    opt = torch.optim.AdamW(W.parameters(), lr=lr)
    for _ in range(epochs):
        loss = F.cross_entropy(W(Xs), ys); opt.zero_grad(); loss.backward(); opt.step()
    return W, mu, sd


@torch.no_grad()
def _score(W, mu, sd, X, y, device=DEVICE):
    return (W(((X - mu) / sd).to(device)).argmax(-1).cpu() == y).float().mean().item()


def _spanend_hidden(eval_set, H, lg, L, level):
    n = len(eval_set[lg]["path_of"])
    s = spanend_samples(eval_set[lg], level, range(n))
    h = H[lg][L] - H[lg][L].mean(dim=0, keepdim=True)
    X = torch.stack([h[t, p] for t, p, _ in s])
    return X, torch.tensor([l for _, _, l in s]), torch.tensor([t for t, _, _ in s])


def _disc_subspace(X, y, K):
    """Orthonormal basis (d x K-1) of the span of centered class means."""
    assert all((y == k).any() for k in range(K)), "class missing from a split"
    M = torch.stack([X[y == k].mean(0) for k in range(K)])
    M = M - M.mean(0, keepdim=True)
    _, _, Vt = torch.linalg.svd(M, full_matrices=False)
    return Vt[:K - 1].T


def _overlap(A, B):
    return ((A.T @ B) ** 2).sum().item() / A.shape[1]


def silent(fn, *a, **k):
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)


def agg(v):
    import numpy as np
    v = np.array(v, float)
    return f"{np.nanmean(v):+.2f}\u00b1{np.nanstd(v):.2f}" if np.isfinite(v).any() else "  nan  "


## 13. Instrument 1 (primary) — terminal-recovery probe

The direct ε-floor readout: at every position where a syncretic surface
token was observed, can a linear probe on the model's own hidden state
recover the TRUE terminal identity? Chance = majority class among the true
terminals sharing that surface token. This is the same held-out-tree
methodology as Exp 6's readout ceiling probe, restated at the terminal
level -- and it doesn't suffer the token-entropy confound that raw CE does
(syncretic positions are trivially easier to predict in raw next-token CE
purely because fewer surface tokens are possible; this probe asks a
structure question -- "what's the NT/terminal here" -- that's equally hard
for every language regardless of surface entropy).

If accuracy stays near chance across the WHOLE capacity range (8→96), the
model never resolves the ambiguity -- that's the ε floor. If it climbs to
ceiling by d=96, capacity fixed what syncretism broke, and Section 5's
pre-flight check told you why (context fully disambiguates within a short
local window).

In [22]:
def terminal_probe_samples(pack, surface_map, ids):
    """(tree, position, true_terminal) for every LEAF position whose surface
    token is ambiguous under surface_map (maps from >1 true terminal)."""
    ambiguous_surfaces = {}
    for true_t, surf_t in surface_map.items():
        ambiguous_surfaces.setdefault(surf_t, set()).add(true_t)
    ambiguous_surfaces = {s for s, ts in ambiguous_surfaces.items() if len(ts) > 1}
    out = []
    idx, tt, mask = pack["idx"], pack["true_term"], pack["mask"]
    for t in ids:
        for p in range(idx.shape[1]):
            if not mask[t, p]:
                continue
            if int(idx[t, p].item()) in ambiguous_surfaces:
                out.append((t, p, int(tt[t, p].item())))
    return out


def run_terminal_recovery_probe(model, rules, d_model, seq_len, eps_level,
                                n_eval_trees=300, eval_seed=999, test_frac=0.2,
                                layers=(6, 8, 10, 11), save_path=None):
    surf = {"L_A": make_surface_map(0), "L_B": make_surface_map(eps_level), "L_C": make_surface_map(eps_level)}
    eval_set = build_matched_eval_set_eps(rules, n_eval_trees, eval_seed, seq_len, eps_level)
    train_ids, test_ids = split_trees(n_eval_trees, test_frac=test_frac, seed=0)
    H = {lg: extract_hidden_states(model, eval_set[lg]["idx"], list(layers)) for lg in LANG_NAMES}
    results = {}
    print(f"\n[d={d_model}, eps={eps_level}] terminal-recovery probe (own-language, held-out trees)")
    print(f"{'L':>2s} {'lang':>4s} {'n_train':>8s} {'n_test':>7s} {'chance':>7s} {'acc':>6s} {'above-chance':>12s}")
    for lg in ("L_B", "L_C"):
        smp_train = terminal_probe_samples(eval_set[lg], surf[lg], train_ids)
        smp_test = terminal_probe_samples(eval_set[lg], surf[lg], test_ids)
        if not smp_train or not smp_test:
            print(f"   {lg}: no ambiguous positions found (eps_level={eps_level} may be 0)")
            continue
        y_test = torch.tensor([lab for _, _, lab in smp_test])
        chance = torch.bincount(y_test).max().item() / len(y_test)
        for L in layers:
            def feats(samples):
                h = H[lg][L] - H[lg][L].mean(dim=0, keepdim=True)
                X = torch.stack([h[t, p] for t, p, _ in samples])
                y = torch.tensor([lab for _, _, lab in samples])
                return X, y
            Xtr, ytr = feats(smp_train)
            Xte, yte = feats(smp_test)
            n_class = int(max(ytr.max(), yte.max())) + 1
            W, mu, sd = _fit_logreg(Xtr, ytr, n_class)
            acc = _score(W, mu, sd, Xte, yte)
            results[(lg, L)] = dict(chance=chance, acc=acc, n_train=len(smp_train), n_test=len(smp_test))
            print(f"{L:>2d} {lg:>4s} {len(smp_train):>8d} {len(smp_test):>7d} {chance:7.3f} {acc:6.3f} "
                  f"{acc - chance:+12.3f}")
    if save_path:
        import pickle
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        pickle.dump(results, open(save_path, "wb"))
    return results


*Running the full `eps_level in (1,2)` x `d` sweep in one cell hit a GPU-contention interrupt after only `eps_level=1, d=8` completed. Restarted below per-checkpoint instead of as one long loop. **`eps_level=2`'s shared-vs-dedicated terminal-recovery comparison was never completed** -- the design was abandoned (see the outcome note at the end of this notebook) before it was rerun.*

In [23]:
# ---- terminal-recovery probe on the budget-matched DEDICATED checkpoints, eps_level=1 ----
# Same instrument as Section 13, just pointed at bm_ckpt_path instead of eps_shared_path.
# Compared at matched parameter count: d'=8 vs shared d=16, d'=16 vs shared d=32, d'=56 vs shared d=96.

dedic_term_recovery_results = {}
for d in BM_SHARED_DS:                      # [16, 32, 96]
    dprime = BM_WIDTHS[d]
    for lang in ("L_B", "L_C"):
        for seed in SEEDS3:
            m = silent(load_shared_ckpt, bm_ckpt_path(1, lang, dprime, seed), dprime, SEQ_LEN_CHECK, n_head=N_HEAD)
            # run_terminal_recovery_probe evaluates both L_B and L_C internally;
            # we only keep this dedicated model's own language's rows.
            full = silent(run_terminal_recovery_probe, m, check_rules, dprime, SEQ_LEN_CHECK, eps_level=1)
            dedic_term_recovery_results[(d, dprime, lang, seed)] = {
                k: v for k, v in full.items() if k[0] == lang
            }
            del m; torch.cuda.empty_cache()
            print(f"done dedicated d'={dprime} (matched to shared d={d}) lang={lang} seed={seed}", flush=True)


done dedicated d'=8 (matched to shared d=16) lang=L_B seed=0
done dedicated d'=8 (matched to shared d=16) lang=L_B seed=7
done dedicated d'=8 (matched to shared d=16) lang=L_B seed=13
done dedicated d'=8 (matched to shared d=16) lang=L_C seed=0
done dedicated d'=8 (matched to shared d=16) lang=L_C seed=7
done dedicated d'=8 (matched to shared d=16) lang=L_C seed=13
done dedicated d'=16 (matched to shared d=32) lang=L_B seed=0
done dedicated d'=16 (matched to shared d=32) lang=L_B seed=7
done dedicated d'=16 (matched to shared d=32) lang=L_B seed=13
done dedicated d'=16 (matched to shared d=32) lang=L_C seed=0
done dedicated d'=16 (matched to shared d=32) lang=L_C seed=7
done dedicated d'=16 (matched to shared d=32) lang=L_C seed=13
done dedicated d'=56 (matched to shared d=96) lang=L_B seed=0
done dedicated d'=56 (matched to shared d=96) lang=L_B seed=7
done dedicated d'=56 (matched to shared d=96) lang=L_B seed=13
done dedicated d'=56 (matched to shared d=96) lang=L_C seed=0
done dedi

In [25]:
for (eps_level, d, seed) in missing:
    m = silent(load_shared_ckpt, eps_shared_path(eps_level, d, seed), d, SEQ_LEN_CHECK, n_head=N_HEAD)
    term_recovery_results[(eps_level, d, seed)] = silent(
        run_terminal_recovery_probe, m, check_rules, d, SEQ_LEN_CHECK, eps_level
    )
    del m; torch.cuda.empty_cache()
    print(f"patched eps={eps_level} d={d} seed={seed}", flush=True)

patched eps=1 d=16 seed=0
patched eps=1 d=16 seed=7
patched eps=1 d=16 seed=13
patched eps=1 d=32 seed=0
patched eps=1 d=32 seed=7
patched eps=1 d=32 seed=13
patched eps=1 d=96 seed=0
patched eps=1 d=96 seed=7
patched eps=1 d=96 seed=13


In [28]:
print("\n=== terminal-recovery accuracy above chance: dedicated vs. shared, matched params, eps_level=1 ===")
print(f"{'d(shared)':>9s} {'d_prime':>8s} {'lang':>4s} {'L':>2s} "
      f"{'dedic acc':>10s} {'shared acc':>11s} {'dedic-shared':>13s}")
for d in BM_SHARED_DS:
    dprime = BM_WIDTHS[d]
    for lang in ("L_B", "L_C"):
        for L in (6, 8, 10, 11):
            dedic_vals = []
            for seed in SEEDS3:
                entry = dedic_term_recovery_results.get((d, dprime, lang, seed), {})
                r = entry.get((lang, L))
                if r is not None:
                    dedic_vals.append(r["acc"] - r["chance"])
            shared_vals = []
            for seed in SEEDS3:
                entry = term_recovery_results.get((1, d, seed), {})
                r = entry.get((lang, L))
                if r is not None:
                    shared_vals.append(r["acc"] - r["chance"])
            if dedic_vals and shared_vals:
                dm = sum(dedic_vals) / len(dedic_vals)
                sm = sum(shared_vals) / len(shared_vals)
                print(f"{d:>9d} {dprime:>8d} {lang:>4s} {L:>2d} {dm:>10.3f} {sm:>11.3f} {dm - sm:>+13.3f}")
            else:
                print(f"{d:>9d} {dprime:>8d} {lang:>4s} {L:>2d}    [skipped]")


=== terminal-recovery accuracy above chance: dedicated vs. shared, matched params, eps_level=1 ===
d(shared)  d_prime lang  L  dedic acc  shared acc  dedic-shared
       16        8  L_B  6      0.029       0.009        +0.020
       16        8  L_B  8      0.018       0.030        -0.011
       16        8  L_B 10      0.029       0.039        -0.011
       16        8  L_B 11      0.037       0.030        +0.007
       16        8  L_C  6      0.036       0.049        -0.012
       16        8  L_C  8      0.048       0.056        -0.008
       16        8  L_C 10      0.016       0.042        -0.026
       16        8  L_C 11      0.021       0.043        -0.022
       32       16  L_B  6      0.014       0.050        -0.035
       32       16  L_B  8      0.038       0.049        -0.011
       32       16  L_B 10      0.028       0.051        -0.023
       32       16  L_B 11      0.028       0.054        -0.026
       32       16  L_C  6      0.063       0.064        -0.001
    

In [29]:
def surface_recovery_sanity_check(model, rules, d_model, seq_len, eps_level, lang="L_B",
                                   layer=11, n_eval_trees=300, eval_seed=999, test_frac=0.2):
    eval_set = build_matched_eval_set_eps(rules, n_eval_trees, eval_seed, seq_len, eps_level)
    train_ids, test_ids = split_trees(n_eval_trees, test_frac=test_frac, seed=0)
    H = extract_hidden_states(model, eval_set[lang]["idx"], [layer])
    idx, mask = eval_set[lang]["idx"], eval_set[lang]["mask"]

    def samples(ids):
        return [(t, p, int(idx[t, p].item())) for t in ids for p in range(idx.shape[1]) if mask[t, p]]

    smp_train, smp_test = samples(train_ids), samples(test_ids)
    h = H[layer] - H[layer].mean(dim=0, keepdim=True)
    Xtr = torch.stack([h[t, p] for t, p, _ in smp_train]); ytr = torch.tensor([l for _, _, l in smp_train])
    Xte = torch.stack([h[t, p] for t, p, _ in smp_test]); yte = torch.tensor([l for _, _, l in smp_test])
    n_class = int(max(ytr.max(), yte.max())) + 1
    W, mu, sd = _fit_logreg(Xtr, ytr, n_class)
    acc = _score(W, mu, sd, Xte, yte)
    chance = torch.bincount(yte).max().item() / len(yte)
    print(f"[{lang}, d={d_model}, L={layer}] surface-token recovery: acc={acc:.3f}  chance={chance:.3f}  "
          f"(should be near-100% -- if not, hidden-state/position indexing is misaligned)")

m = silent(load_shared_ckpt, eps_shared_path(1, 96, 0), 96, SEQ_LEN_CHECK, n_head=N_HEAD)
surface_recovery_sanity_check(m, check_rules, 96, SEQ_LEN_CHECK, eps_level=1, lang="L_B", layer=11)
del m; torch.cuda.empty_cache()

[L_B, d=96, L=11] surface-token recovery: acc=0.992  chance=0.478  (should be near-100% -- if not, hidden-state/position indexing is misaligned)


In [30]:
def oracle_context_recovery(rules, eps_level, lang="L_B", K=32, n_eval_trees=300,
                             eval_seed=999, test_frac=0.2, seq_len=SEQ_LEN_CHECK):
    eval_set = build_matched_eval_set_eps(rules, n_eval_trees, eval_seed, seq_len, eps_level)
    train_ids, test_ids = split_trees(n_eval_trees, test_frac=test_frac, seed=0)
    surf = make_surface_map(eps_level)
    ambiguous = {s for s, ts in {s: {t for t, s2 in surf.items() if s2 == s} for s in set(surf.values())}.items() if len(ts) > 1}
    idx, tt, mask = eval_set[lang]["idx"], eval_set[lang]["true_term"], eval_set[lang]["mask"]

    def samples(ids):
        out = []
        for t in ids:
            for p in range(idx.shape[1]):
                if mask[t, p] and int(idx[t, p].item()) in ambiguous:
                    ctx = idx[t, max(0, p - K):p].tolist()
                    ctx = [-1] * (K - len(ctx)) + ctx
                    out.append((ctx, int(tt[t, p].item())))
        return out

    smp_train, smp_test = samples(train_ids), samples(test_ids)
    if not smp_train or not smp_test:
        print(f"{lang}, eps={eps_level}: no ambiguous positions found"); return
    Xtr = torch.tensor([c for c, _ in smp_train], dtype=torch.float); ytr = torch.tensor([l for _, l in smp_train])
    Xte = torch.tensor([c for c, _ in smp_test], dtype=torch.float); yte = torch.tensor([l for _, l in smp_test])
    n_class = int(max(ytr.max(), yte.max())) + 1
    W, mu, sd = _fit_logreg(Xtr, ytr, n_class)
    acc = _score(W, mu, sd, Xte, yte)
    chance = torch.bincount(yte).max().item() / len(yte)
    print(f"[{lang}, eps={eps_level}, K={K}] oracle (raw context, no model): "
          f"acc={acc:.3f} chance={chance:.3f} above-chance={acc-chance:+.3f} n_test={len(yte)}")

oracle_context_recovery(check_rules, eps_level=1, lang="L_B", K=32)
oracle_context_recovery(check_rules, eps_level=1, lang="L_C", K=32)

[L_B, eps=1, K=32] oracle (raw context, no model): acc=0.679 chance=0.668 above-chance=+0.011 n_test=4184
[L_C, eps=1, K=32] oracle (raw context, no model): acc=0.691 chance=0.668 above-chance=+0.023 n_test=4184


In [32]:
def oracle_context_recovery_onehot(rules, eps_level, lang="L_B", K=32, n_eval_trees=300,
                                    eval_seed=999, test_frac=0.2, seq_len=SEQ_LEN_CHECK):
    eval_set = build_matched_eval_set_eps(rules, n_eval_trees, eval_seed, seq_len, eps_level)
    train_ids, test_ids = split_trees(n_eval_trees, test_frac=test_frac, seed=0)
    surf = make_surface_map(eps_level)
    ambiguous = {s for s, ts in {s: {t for t, s2 in surf.items() if s2 == s} for s in set(surf.values())}.items() if len(ts) > 1}
    idx, tt, mask = eval_set[lang]["idx"], eval_set[lang]["true_term"], eval_set[lang]["mask"]

    def samples(ids):
        out = []
        for t in ids:
            for p in range(idx.shape[1]):
                if mask[t, p] and int(idx[t, p].item()) in ambiguous:
                    ctx = idx[t, max(0, p - K):p].tolist()
                    ctx = [-1] * (K - len(ctx)) + ctx
                    out.append((ctx, int(tt[t, p].item())))
        return out

    smp_train, smp_test = samples(train_ids), samples(test_ids)
    if not smp_train or not smp_test:
        print(f"{lang}, eps={eps_level}: no ambiguous positions found"); return

    # size the vocab from what's ACTUALLY in the sampled contexts, not an assumption
    all_vals = {v for ctx, _ in smp_train + smp_test for v in ctx}
    max_val = max(all_vals)              # highest real token id seen (could include BOS ids)
    n_cols_per_slot = max_val + 2         # +1 for 0-indexing, +1 for the padding(-1) slot

    def onehot(ctx_list):
        X = torch.zeros(len(ctx_list), K * n_cols_per_slot)
        for i, ctx in enumerate(ctx_list):
            for k, val in enumerate(ctx):
                col = val + 1 if val >= 0 else 0
                X[i, k * n_cols_per_slot + col] = 1.0
        return X

    Xtr = onehot([c for c, _ in smp_train]); ytr = torch.tensor([l for _, l in smp_train])
    Xte = onehot([c for c, _ in smp_test]); yte = torch.tensor([l for _, l in smp_test])
    n_class = int(max(ytr.max(), yte.max())) + 1
    W, mu, sd = _fit_logreg(Xtr, ytr, n_class)
    acc = _score(W, mu, sd, Xte, yte)
    chance = torch.bincount(yte).max().item() / len(yte)
    print(f"[{lang}, eps={eps_level}, K={K}, one-hot, vocab={max_val+1}] oracle: "
          f"acc={acc:.3f} chance={chance:.3f} above-chance={acc-chance:+.3f} n_test={len(yte)}")

oracle_context_recovery_onehot(check_rules, eps_level=1, lang="L_B", K=32)
oracle_context_recovery_onehot(check_rules, eps_level=1, lang="L_C", K=32)

[L_B, eps=1, K=32, one-hot, vocab=6] oracle: acc=0.697 chance=0.668 above-chance=+0.030 n_test=4184
[L_C, eps=1, K=32, one-hot, vocab=7] oracle: acc=0.712 chance=0.668 above-chance=+0.044 n_test=4184


In [33]:
def preflight_repeated_only(rules, eps_level, lang="L_B", K=32, n_trees=20_000, seed=0):
    surf = make_surface_map(eps_level)
    ambiguous_surfaces = {s for s, ts in {s: {t for t, s2 in surf.items() if s2 == s} for s in set(surf.values())}.items() if len(ts) > 1}
    rng = random.Random(seed)
    revset = LANGS[lang]
    ctx_to_true = {}

    def walk(node, out):
        if isinstance(node, int):
            out.append(node); return
        (level, idx), children = node
        order = list(reversed(children)) if level in revset else children
        for c in order:
            walk(c, out)

    for _ in range(n_trees):
        tree = sample_tree(rules, rng)
        toks_true = []
        walk(tree, toks_true)
        surf_stream = [surf[t] for t in toks_true]
        for p, (true_t, s) in enumerate(zip(toks_true, surf_stream)):
            if s not in ambiguous_surfaces:
                continue
            ctx = tuple(surf_stream[max(0, p - K):p])
            ctx_to_true.setdefault(ctx, set()).add(true_t)

    n_once = sum(1 for v in ctx_to_true.values() if True)  # placeholder, real split below
    counts = {}
    rng2 = random.Random(seed)
    occ_count = {}
    # recount occurrences (not just unique resolutions) to split by repeat frequency
    occ = {}
    rng3 = random.Random(seed)
    for _ in range(n_trees):
        tree = sample_tree(rules, rng3)
        toks_true = []
        walk(tree, toks_true)
        surf_stream = [surf[t] for t in toks_true]
        for p, (true_t, s) in enumerate(zip(toks_true, surf_stream)):
            if s not in ambiguous_surfaces:
                continue
            ctx = tuple(surf_stream[max(0, p - K):p])
            occ[ctx] = occ.get(ctx, 0) + 1

    seen_once = {c for c, n in occ.items() if n == 1}
    seen_multi = {c for c, n in occ.items() if n >= 2}
    n_ambig_once = sum(1 for c in seen_once if c in ctx_to_true and len(ctx_to_true[c]) > 1)
    n_ambig_multi = sum(1 for c in seen_multi if c in ctx_to_true and len(ctx_to_true[c]) > 1)

    print(f"[{lang}, eps={eps_level}, K={K}] contexts seen once: {len(seen_once)} (ambiguous: {n_ambig_once}, "
          f"frac {n_ambig_once/max(1,len(seen_once)):.3f})")
    print(f"[{lang}, eps={eps_level}, K={K}] contexts seen >=2x: {len(seen_multi)} (ambiguous: {n_ambig_multi}, "
          f"frac {n_ambig_multi/max(1,len(seen_multi)):.3f})  <-- this is the honest resolvability number")

preflight_repeated_only(check_rules, eps_level=1, lang="L_B", K=32)
preflight_repeated_only(check_rules, eps_level=1, lang="L_C", K=32)

[L_B, eps=1, K=32] contexts seen once: 1591918 (ambiguous: 0, frac 0.000)
[L_B, eps=1, K=32] contexts seen >=2x: 9750 (ambiguous: 3071, frac 0.315)  <-- this is the honest resolvability number
[L_C, eps=1, K=32] contexts seen once: 1593351 (ambiguous: 0, frac 0.000)
[L_C, eps=1, K=32] contexts seen >=2x: 9634 (ambiguous: 3756, frac 0.390)  <-- this is the honest resolvability number


## 14. Instrument 2 (primary) — CE gap: shared vs. budget-matched dedicated

Instantiates the derivation's L_S(N) − L_D(N) directly. `CE_shared` and
`CE_dedicated` are both computed on FRESH held-out data (mixed-row format
for shared, own-format for dedicated), matching Exp 6's established
methodology for the primary reported CE numbers.

Positive gap = sharing costs more than budget-matched dedicated (dedicated
wins). Negative = sharing wins (capacity-pooling benefit). The crossover
`d*` where the sign flips is the derivation's N* -- read directly off this
table, no representation-space instrument required.

In [17]:
def ce_gap_shared_vs_dedicated(eps_level, d_shared, dprime, seed, fresh_seed=98765):
    shared_m = silent(load_shared_ckpt, eps_shared_path(eps_level, d_shared, seed), d_shared, SEQ_LEN_CHECK, n_head=N_HEAD)
    fresh_shared = make_shared_tree_mix_eps(check_rules, n_trees=20_000, seed=fresh_seed,
                                            seq_len=SEQ_LEN_CHECK, eps_level=eps_level, mix=MIX)
    out = {}
    for lg in ("L_B", "L_C"):
        ce_shared = first_tree_ce(shared_m, fresh_shared, BOS[lg], torch.tensor(list(BOS.values())))
        if ce_shared is None:
            raise RuntimeError(f"no rows starting with BOS[{lg}] in the fresh draw of size 20000 -- "
                                f"increase n_trees further, or check MIX/BOS")
        dedic_m = silent(load_shared_ckpt, bm_ckpt_path(eps_level, lg, dprime, seed), dprime, SEQ_LEN_CHECK, n_head=N_HEAD)
        fresh_solo = make_monolingual_dataset_eps(check_rules, lg, n_trees=3000, seed=fresh_seed,
                                                  seq_len=SEQ_LEN_CHECK, eps_level=eps_level)
        ce_dedic = mean_ce(dedic_m, fresh_solo)
        out[lg] = dict(ce_shared=ce_shared, ce_dedicated=ce_dedic, gap=ce_shared - ce_dedic)
        del dedic_m; torch.cuda.empty_cache()
    del shared_m; torch.cuda.empty_cache()
    return out


print(f"{'eps':>4s} {'d':>4s} {'d_prime':>8s} {'lang':>4s} {'CE_shared':>10s} {'CE_dedic':>10s} {'gap':>8s}")
ce_gap_results = {}
for eps_level in (0, 1):
    for d in BM_SHARED_DS:
        dprime = BM_WIDTHS[d]
        # eps=0 uses the shared checkpoint from Exp 6's original sweep if you have it at this d;
        # otherwise this cell will fail to find eps_shared_path(0, d, seed) -- eps=0 shared models
        # were not retrained in Section 9 (see its markdown). Point SHARED_D0_PATH_FN at your
        # existing Exp 6 checkpoints if you're running the eps=0 budget-matched comparison.
        shared_path_fn = eps_shared_path if eps_level > 0 else eps_shared_path
        for seed in SEEDS3:
            try:
                r = ce_gap_shared_vs_dedicated(eps_level, d, dprime, seed)
            except FileNotFoundError as e:
                print(f"  [skip] eps={eps_level} d={d} seed={seed}: {e}")
                continue
            ce_gap_results[(eps_level, d, seed)] = r
            for lg in ("L_B", "L_C"):
                g = r[lg]
                print(f"{eps_level:>4d} {d:>4d} {dprime:>8d} {lg:>4s} {g['ce_shared']:10.4f} "
                      f"{g['ce_dedicated']:10.4f} {g['gap']:+8.4f}")

print("\n=== summary: mean CE gap over seeds (negative = sharing wins) ===")
print(f"{'eps':>4s} {'d':>4s} {'lang':>4s} {'mean gap':>10s}")
for eps_level in (0, 1):
    for d in BM_SHARED_DS:
        for lg in ("L_B", "L_C"):
            vals = [ce_gap_results[(eps_level, d, s)][lg]["gap"] for s in SEEDS3
                    if (eps_level, d, s) in ce_gap_results]
            if vals:
                print(f"{eps_level:>4d} {d:>4d} {lg:>4s} {sum(vals)/len(vals):+10.4f}")

 eps    d  d_prime lang  CE_shared   CE_dedic      gap
  [skip] eps=0 d=16 seed=0: [Errno 2] No such file or directory: './exp6b_syncretism/shared_eps0_d16_seed0.pt'
  [skip] eps=0 d=16 seed=7: [Errno 2] No such file or directory: './exp6b_syncretism/shared_eps0_d16_seed7.pt'
  [skip] eps=0 d=16 seed=13: [Errno 2] No such file or directory: './exp6b_syncretism/shared_eps0_d16_seed13.pt'
  [skip] eps=0 d=32 seed=0: [Errno 2] No such file or directory: './exp6b_syncretism/shared_eps0_d32_seed0.pt'
  [skip] eps=0 d=32 seed=7: [Errno 2] No such file or directory: './exp6b_syncretism/shared_eps0_d32_seed7.pt'
  [skip] eps=0 d=32 seed=13: [Errno 2] No such file or directory: './exp6b_syncretism/shared_eps0_d32_seed13.pt'
  [skip] eps=0 d=96 seed=0: [Errno 2] No such file or directory: './exp6b_syncretism/shared_eps0_d96_seed0.pt'
  [skip] eps=0 d=96 seed=7: [Errno 2] No such file or directory: './exp6b_syncretism/shared_eps0_d96_seed7.pt'
  [skip] eps=0 d=96 seed=13: [Errno 2] No such file o

## 15. Instrument 3 (secondary) — interchange intervention

Causal alternative to class-mean subspace overlap. Same underlying tree,
two languages: run A's forward pass, then run the target language's (B or
C's) forward pass but PATCH its residual stream at `layer`, at the span-end
position of a given level-`level` NT instance, with A's actual activation
for the SAME NT instance (matched via `inst`, which is language-invariant
by construction -- see Section 2). Re-score the target's own held-out-tree
readout on the patched activations.

If patched accuracy ≈ unpatched (own-language) accuracy, A's representation
is functionally interchangeable with the target's own -- sharing. If it
drops toward chance, A's representation actively hurts the target's
read-out -- separation. Unlike class-mean overlap, this doesn't have the
thin-dynamic-range problem (it substitutes real activations, not zeros) and
doesn't need a train/test split of its own beyond what the readout probe
already uses.

**Calibrate with A′ first** (Section 11): A→A′ interchange should show
patched ≈ baseline (they compute identically). Any interchange result on
A→B/A→C should be read relative to that ceiling.

In [20]:
def interchange_intervention(model, eval_set, src_lang, tgt_lang, layer, level, train_ids, test_ids):
    """eval_set must contain BOTH src_lang and tgt_lang's packs (built from
    the same underlying sample_tree_indexed trees, e.g. via
    build_matched_eval_set_eps or make_shared_tree_mix_aprime's eval-set
    counterpart)."""
    H_src = extract_hidden_states(model, eval_set[src_lang]["idx"], [layer])
    H_tgt = extract_hidden_states(model, eval_set[tgt_lang]["idx"], [layer])

    X_all, y_all, t_all = _spanend_hidden(eval_set, {tgt_lang: H_tgt}, tgt_lang, layer, level)
    t_list = t_all.tolist()
    train_mask = torch.tensor([tid in train_ids for tid in t_list])
    test_mask = torch.tensor([tid in test_ids for tid in t_list])
    n_class = int(y_all.max()) + 1
    W, mu, sd = _fit_logreg(X_all[train_mask], y_all[train_mask], n_class)
    baseline_acc = _score(W, mu, sd, X_all[test_mask], y_all[test_mask])

    pos_map_src = spanend_position_map(eval_set[src_lang], level, test_ids)
    pos_map_tgt = spanend_position_map(eval_set[tgt_lang], level, test_ids)
    h_src = H_src[layer]
    h_tgt_mean = H_tgt[layer].mean(dim=0, keepdim=True)
    patched_X, patched_y = [], []
    for (t, key), p_tgt in pos_map_tgt.items():
        if (t, key) not in pos_map_src:
            continue
        p_src = pos_map_src[(t, key)]
        patched_X.append(h_src[t, p_src] - h_tgt_mean[0, p_tgt])
        patched_y.append(key[1])
    if not patched_X:
        return dict(baseline=baseline_acc, patched=float("nan"), n=0)
    patched_X = torch.stack(patched_X)
    patched_y = torch.tensor(patched_y)
    patched_acc = _score(W, mu, sd, patched_X, patched_y)
    return dict(baseline=baseline_acc, patched=patched_acc, n=len(patched_y))


# --- A' calibration ceiling (uses the aprime eval set at eps_level=1) ---
def build_matched_eval_set_aprime(rules, n_trees, seed, seq_len, eps_level, max_leaves=None):
    surf = {"L_A": make_surface_map(0), "L_A_PRIME": make_surface_map(0),
            "L_B": make_surface_map(eps_level), "L_C": make_surface_map(eps_level)}
    rng = random.Random(seed)
    cap = seq_len - 1
    trees_kept = []
    while len(trees_kept) < n_trees:
        tree = sample_tree_indexed(rules, rng)
        n_leaves = len(linearize_with_paths(tree, LANGS_A["L_A"]))
        if n_leaves > cap or (max_leaves is not None and n_leaves > max_leaves):
            continue
        trees_kept.append(tree)
    out = {}
    for lang in LANG_NAMES_A:
        idx = torch.zeros(n_trees, seq_len, dtype=torch.long)
        true_term = -torch.ones(n_trees, seq_len, dtype=torch.long)
        mask = torch.zeros(n_trees, seq_len, dtype=torch.bool)
        leaf_id = -torch.ones(n_trees, seq_len, dtype=torch.long)
        path_of = []
        for t, tree in enumerate(trees_kept):
            linearized = linearize_with_paths(tree, LANGS_A[lang])
            idx[t, 0] = BOS_A[lang]
            per_tree_paths = {}
            for pos, (tok, lid, pth) in enumerate(linearized):
                idx[t, pos + 1] = surf[lang][tok]
                true_term[t, pos + 1] = tok
                mask[t, pos + 1] = True
                leaf_id[t, pos + 1] = lid
                per_tree_paths[lid] = pth
            path_of.append(per_tree_paths)
        out[lang] = dict(idx=idx, true_term=true_term, mask=mask, leaf_id=leaf_id, path_of=path_of)
    return out


N_EVAL_TREES_IX = 300
EVAL_SEED_IX = 999
interchange_results = {}
for d in APRIME_D_LIST:
    for seed in SEEDS3:
        m = silent(load_shared_ckpt, aprime_path(d, seed), d, SEQ_LEN_CHECK, vocab_size=VOCAB_A, n_head=N_HEAD)
        eval_set_a = build_matched_eval_set_aprime(check_rules, N_EVAL_TREES_IX, EVAL_SEED_IX, SEQ_LEN_CHECK, eps_level=1)
        train_ids, test_ids = split_trees(N_EVAL_TREES_IX, test_frac=0.2, seed=0)
        for layer in (8, 11):
            for level in (4, 5, 6):
                r = interchange_intervention(m, eval_set_a, "L_A", "L_A_PRIME", layer, level, train_ids, test_ids)
                interchange_results[("calib", d, seed, layer, level)] = r
        del m; torch.cuda.empty_cache()
        print(f"done A-A' calibration d={d} seed={seed}", flush=True)

print("\n=== A -> A' calibration ceiling (patched should be ~= baseline) ===")
print(f"{'d':>4s} {'L':>2s} {'lvl':>3s} {'baseline':>9s} {'patched':>8s} {'n':>5s}")
for d in APRIME_D_LIST:
    for layer in (8, 11):
        for level in (4, 5, 6):
            vals = [interchange_results[("calib", d, s, layer, level)] for s in SEEDS3]
            base = sum(v["baseline"] for v in vals) / len(vals)
            patch = sum(v["patched"] for v in vals if v["n"] > 0) / max(1, sum(1 for v in vals if v["n"] > 0))
            print(f"{d:>4d} {layer:>2d} {level:>3d} {base:9.3f} {patch:8.3f} {vals[0]['n']:>5d}")

print("\nOnce this ceiling looks right (patched ~ baseline), rerun interchange_intervention with")
print("src_lang='L_A', tgt_lang='L_B' or 'L_C' on eval_set built from build_matched_eval_set_eps")
print("at eps_level in (1, 2), same d/seed sweep, to get the real A->B / A->C interchange results.")


done A-A' calibration d=16 seed=0
done A-A' calibration d=16 seed=7
done A-A' calibration d=16 seed=13
done A-A' calibration d=32 seed=0
done A-A' calibration d=32 seed=7
done A-A' calibration d=32 seed=13
done A-A' calibration d=96 seed=0
done A-A' calibration d=96 seed=7
done A-A' calibration d=96 seed=13

=== A -> A' calibration ceiling (patched should be ~= baseline) ===
   d  L lvl  baseline  patched     n
  16  8   4     0.700    0.698   838
  16  8   5     0.810    0.808  1929
  16  8   6     0.907    0.907  4756
  16 11   4     0.658    0.656   838
  16 11   5     0.739    0.740  1929
  16 11   6     0.843    0.843  4756
  32  8   4     0.812    0.813   838
  32  8   5     0.875    0.875  1929
  32  8   6     0.905    0.905  4756
  32 11   4     0.785    0.783   838
  32 11   5     0.858    0.859  1929
  32 11   6     0.874    0.875  4756
  96  8   4     0.805    0.807   838
  96  8   5     0.867    0.865  1929
  96  8   6     0.923    0.922  4756
  96 11   4     0.828    0.831

## 16. Instrument 4 (secondary) — token-out subspace overlap

Reused from Exp 6 for consistency-checking against the primary instruments
above (not a new claim on its own -- see Exp 6's learnings on why raw
overlap and probe-transfer are noisy). `run_subspace_overlap` here takes a
pre-built `eval_set` directly, so it works on either the plain Exp 6 eval
set or the eps-aware one without duplicating the function.

In [21]:
def run_subspace_overlap(model, eval_set, d_model, layers=(6, 8, 10, 11), levels=(4, 5, 6),
                         n_split=10, tag=""):
    import numpy as np
    H = {lg: extract_hidden_states(model, eval_set[lg]["idx"], list(layers)) for lg in LANG_NAMES}
    rng = np.random.RandomState(0)
    n_eval_trees = eval_set["L_A"]["idx"].shape[0]
    out = {}
    print(f"\n[d={d_model}{tag}] discriminative-subspace overlap (mean cos^2 of principal angles)")
    print(f"{'lvl':>3s} {'L':>2s} {'null':>6s} | {'floorA':>6s} {'floorB':>6s} {'floorC':>6s} | "
          f"{'A-B':>6s} {'A-C':>6s} {'B-C':>6s}")
    for lvl in levels:
        for L in layers:
            D = {lg: _spanend_hidden(eval_set, H, lg, L, lvl) for lg in LANG_NAMES}
            pooled = torch.cat([D[lg][0] for lg in LANG_NAMES])
            mu_p, sd_p = pooled.mean(0), pooled.std(0) + 1e-6
            Z = {lg: ((D[lg][0] - mu_p) / sd_p, D[lg][1], D[lg][2]) for lg in LANG_NAMES}
            K = int(max(Z[lg][1].max() for lg in LANG_NAMES)) + 1
            S = {lg: _disc_subspace(Z[lg][0], Z[lg][1], K) for lg in LANG_NAMES}
            fl = {lg: [] for lg in LANG_NAMES}
            for _ in range(n_split):
                h1 = torch.tensor(rng.permutation(n_eval_trees)[: n_eval_trees // 2])
                for lg in LANG_NAMES:
                    X, y, t = Z[lg]; m = torch.isin(t, h1)
                    fl[lg].append(_overlap(_disc_subspace(X[m], y[m], K), _disc_subspace(X[~m], y[~m], K)))
            fl = {lg: float(np.mean(v)) for lg, v in fl.items()}
            ab = _overlap(S["L_A"], S["L_B"]); ac = _overlap(S["L_A"], S["L_C"]); bc = _overlap(S["L_B"], S["L_C"])
            null = (K - 1) / Z["L_A"][0].shape[1]
            out[(lvl, L)] = dict(null=null, floor=fl, ab=ab, ac=ac, bc=bc)
            print(f"{lvl:>3d} {L:>2d} {null:6.3f} | {fl['L_A']:6.3f} {fl['L_B']:6.3f} {fl['L_C']:6.3f} | "
                  f"{ab:6.3f} {ac:6.3f} {bc:6.3f}")
    return out


def overlap_tokout(model, eval_set, d_model, **kw):
    """Token-out correction: subtract per-(current,previous-surface-token)
    means before computing overlap, as in Exp 6."""
    global _spanend_hidden
    _orig = _spanend_hidden
    def _tokout(eval_set_, H, lg, L, level):
        X, y, t = _orig(eval_set_, H, lg, L, level)
        s = spanend_samples(eval_set_[lg], level, range(len(eval_set_[lg]["path_of"])))
        idx = eval_set_[lg]["idx"]
        key = torch.tensor([int(idx[tt, p]) * 100 + int(idx[tt, p - 1]) for tt, p, _ in s])
        X = X.clone()
        for k in key.unique():
            m = key == k
            X[m] -= X[m].mean(0, keepdim=True)
        return X, y, t
    _spanend_hidden = _tokout
    try:
        return run_subspace_overlap(model, eval_set, d_model, **kw)
    finally:
        _spanend_hidden = _orig


ov_tokout_results = {}
for eps_level in (1, 2):
    for d in EPS_D_LIST:
        for seed in SEEDS3:
            m = silent(load_shared_ckpt, eps_shared_path(eps_level, d, seed), d, SEQ_LEN_CHECK, n_head=N_HEAD)
            eval_set_ov = build_matched_eval_set_eps(check_rules, 300, 999, SEQ_LEN_CHECK, eps_level)
            ov_tokout_results[(eps_level, d, seed)] = silent(
                overlap_tokout, m, eval_set_ov, d, layers=(8, 11), levels=(4, 5, 6)
            )
            del m; torch.cuda.empty_cache()
            print(f"done eps={eps_level} d={d} seed={seed}", flush=True)

print("\n=== chance-corrected overlap with A (mean\u00b1sd over seeds); nan = floor too close to null ===")
for eps_level in (1, 2):
    for L in (8, 11):
        print(f"\\n--- eps={eps_level}, L{L} ---")
        print(f"{'d':>4s} | " + " | ".join(f"lvl{l}: {'A-B':>11s} {'A-C':>11s}" for l in (4, 5, 6)))
        for d in EPS_D_LIST:
            parts = []
            for l in (4, 5, 6):
                ab = [cc_safe(ov_tokout_results[(eps_level, d, s)][(l, L)], "L_B") for s in SEEDS3]
                ac = [cc_safe(ov_tokout_results[(eps_level, d, s)][(l, L)], "L_C") for s in SEEDS3]
                parts.append(f"{agg(ab):>11s} {agg(ac):>11s}")
            print(f"{d:>4d} | " + " | ".join(parts))


done eps=1 d=8 seed=0
done eps=1 d=8 seed=7
done eps=1 d=8 seed=13
done eps=1 d=16 seed=0
done eps=1 d=16 seed=7
done eps=1 d=16 seed=13
done eps=1 d=32 seed=0
done eps=1 d=32 seed=7
done eps=1 d=32 seed=13
done eps=1 d=96 seed=0
done eps=1 d=96 seed=7
done eps=1 d=96 seed=13
done eps=2 d=8 seed=0
done eps=2 d=8 seed=7
done eps=2 d=8 seed=13
done eps=2 d=16 seed=0
done eps=2 d=16 seed=7
done eps=2 d=16 seed=13
done eps=2 d=32 seed=0
done eps=2 d=32 seed=7
done eps=2 d=32 seed=13
done eps=2 d=96 seed=0
done eps=2 d=96 seed=7
done eps=2 d=96 seed=13

=== chance-corrected overlap with A (mean±sd over seeds); nan = floor too close to null ===
\n--- eps=1, L8 ---
   d | lvl4:         A-B         A-C | lvl5:         A-B         A-C | lvl6:         A-B         A-C
   8 |  +0.08±0.54  +1.09±0.12 |  +0.11±0.03  +0.25±0.21 |  +0.43±0.33  +0.35±0.12
  16 |  +0.08±0.12  +0.04±0.09 |  +0.05±0.06  +0.01±0.06 |  +0.05±0.26  -0.04±0.09
  32 |  -0.06±0.05  -0.00±0.04 |  -0.03±0.01  -0.02±0.01 |  -0.01±

## 17. Wrap-up

Reading order for the results once everything's trained:

1. **Section 5 (pre-flight)** should already have told you, before any
   training, whether the collapse is genuinely context-hungry. If it wasn't
   run or came back weak, the results below need that caveat attached.
2. **Section 13 (terminal-recovery probe)** is the primary ε-floor claim:
   does above-chance accuracy plateau below ceiling as d grows, for a
   syncretic language, at a level/layer where the corresponding word-order
   cost (Exp 6's CE gap) has already gone to zero?
3. **Section 14 (CE gap vs. budget-matched)** is the primary crossover
   claim: sign and magnitude of `gap` across `d`, compared between
   `eps_level=0` (pure capacity pooling, no ε) and `eps_level=1` (ε present)
   -- does the crossover point shift?
4. **Sections 15-16** are secondary/consistency checks, read only after 1-3
   look right; a clean interchange/overlap result that contradicts the
   terminal-recovery probe or CE gap is a reason to distrust the
   representation-space instrument, not the primary result (per Exp 6's
   learnings: the functional measures were clean, the geometric ones were
   noisy, and that pattern held up before -- no reason to expect it flipped
   here).

Do not add further conditions after these results come in without
discussing them first.

In [23]:
# ---- entropy floor (not carried over from Exp 6's notebook -- computing it here) ----
import math

def _walk(rules, rng, symbol, level):
    rhs_list = rules[symbol]
    rhs = rng.choice(rhs_list)
    nats = -math.log(rhs_list.count(rhs) / len(rhs_list))
    if level == LEVELS:
        return nats, len(rhs)
    leaves = 0
    for child in rhs:
        n, l = _walk(rules, rng, child, level + 1)
        nats += n; leaves += l
    return nats, leaves

def entropy_per_token(rules, n_trees=20000, seed=0):
    rng = random.Random(seed)
    tot_n = tot_l = 0
    for _ in range(n_trees):
        n, l = _walk(rules, rng, (1, 0), 1)
        tot_n += n; tot_l += l
    return tot_n / tot_l, tot_l / n_trees

H_sh, leaves_sh = entropy_per_token(check_rules)
print(f"grammar entropy H_sh = {H_sh:.4f} nats/token ({leaves_sh:.0f} leaves/tree)")

grammar entropy H_sh = 0.7545 nats/token (238 leaves/tree)


In [24]:
# ---- Convergence check: has eps=1 / eps=2 training at d=96 actually plateaued? ----
# NOTE: H_sh is the TRUE-token entropy floor (same for all languages under Exp 6's
# reversal-only linearization, since reversal is a bijection). It's NOT the right
# floor here: syncretism can only LOWER the achievable per-token entropy for the
# collapsed languages (fewer surface outcomes to distinguish), so H_sh overestimates
# the true floor for the mixed eps>=1 pool. Use it only as a loose upper reference --
# the tail-slope check below is the trustworthy signal.

print(f"{'eps':>4s} {'d':>4s} {'seed':>4s} {'CE @ ~55%':>10s} {'CE @ final 1000':>16s} {'still improving?':>18s}")
for eps_level in (1, 2):
    for seed in SEEDS3:
        ck = torch.load(eps_shared_path(eps_level, 96, seed), map_location="cpu", weights_only=False)
        losses = ck["losses"]
        n = len(losses)
        mid_slice = losses[int(n * 0.55):int(n * 0.56)]
        mid = float(sum(mid_slice) / max(1, len(mid_slice)))
        tail = float(sum(losses[-1000:]) / 1000)
        improving = "yes" if (mid - tail) > 0.003 else "no (looks flat)"
        print(f"{eps_level:>4d} {96:>4d} {seed:>4d} {mid:>10.4f} {tail:>16.4f} {improving:>18s}")

print(f"\n(reference only: H_sh = {H_sh:.4f} -- don't read 'tail > H_sh' as 'not converged' here,")
print(f" since the true achievable floor for L_B/L_C under syncretism is lower than H_sh)")

 eps    d seed  CE @ ~55%  CE @ final 1000   still improving?
   1   96    0     0.7898           0.7809                yes
   1   96    7     0.7848           0.7764                yes
   1   96   13     0.7847           0.7752                yes
   2   96    0     0.7533           0.7423                yes
   2   96    7     0.7555           0.7450                yes
   2   96   13     0.7541           0.7438                yes

(reference only: H_sh = 0.7545 -- don't read 'tail > H_sh' as 'not converged' here,
 since the true achievable floor for L_B/L_C under syncretism is lower than H_sh)


In [25]:
# ---- Quantify how much of the eps=1 crossover could be a budget-match artifact ----
import math

dprimes_sorted = sorted(set(BM_WIDTHS.values()))
ratios = {d: 3 * count_params(BM_WIDTHS[d]) / count_params(d) for d in BM_SHARED_DS}

ce_by_dprime = {}
for dp in dprimes_sorted:
    d_for_dp = [d for d in BM_SHARED_DS if BM_WIDTHS[d] == dp][0]
    vals = []
    for lg in ("L_B", "L_C"):
        vals += [ce_gap_results[(1, d_for_dp, s)][lg]["ce_dedicated"] for s in SEEDS3
                  if (1, d_for_dp, s) in ce_gap_results]
    ce_by_dprime[dp] = sum(vals) / len(vals)

print(f"{'d_prime':>8s} {'params':>10s} {'CE_dedic (eps=1, mean L_B+L_C)':>32s}")
for dp in dprimes_sorted:
    print(f"{dp:>8d} {count_params(dp):>10,d} {ce_by_dprime[dp]:>32.4f}")

slopes = []
print(f"\n{'segment':>10s} {'d(CE)/d(log params)':>22s}")
for a, b in zip(dprimes_sorted[:-1], dprimes_sorted[1:]):
    dlogp = math.log(count_params(b)) - math.log(count_params(a))
    slope = (ce_by_dprime[b] - ce_by_dprime[a]) / dlogp
    slopes.append(slope)
    print(f"{f'{a}->{b}':>10s} {slope:>22.4f}")

print(f"\n{'d':>4s} {'d_prime':>8s} {'ratio':>7s} {'predicted CE shift if ratio=1':>30s} {'observed eps=1 gap':>20s}")
for d in BM_SHARED_DS:
    dp = BM_WIDTHS[d]
    ratio = ratios[d]
    idx = dprimes_sorted.index(dp)
    local_slope = slopes[max(0, idx - 1)] if idx > 0 else slopes[0]
    predicted_shift = -local_slope * math.log(ratio)
    obs = []
    for lg in ("L_B", "L_C"):
        obs += [ce_gap_results[(1, d, s)][lg]["gap"] for s in SEEDS3 if (1, d, s) in ce_gap_results]
    obs_mean = sum(obs) / len(obs)
    print(f"{d:>4d} {dp:>8d} {ratio:>7.3f} {predicted_shift:>30.4f} {obs_mean:>20.4f}")

print("\nIf |predicted_shift| is comparable to or larger than |observed gap|, the ratio mismatch")
print("alone could produce (or reverse) the crossover -- treat it as unconfirmed until the")
print("budget-matched widths are corrected and this comparison is rerun.")

 d_prime     params   CE_dedic (eps=1, mean L_B+L_C)
       8     10,592                           0.7942
      16     39,616                           0.7717
      56    461,216                           0.7714

   segment    d(CE)/d(log params)
     8->16                -0.0170
    16->56                -0.0001

   d  d_prime   ratio  predicted CE shift if ratio=1   observed eps=1 gap
  16        8   0.802                        -0.0037              -0.0218
  32       16   0.777                        -0.0043              -0.0068
  96       56   1.030                         0.0000               0.0054

If |predicted_shift| is comparable to or larger than |observed gap|, the ratio mismatch
alone could produce (or reverse) the crossover -- treat it as unconfirmed until the
budget-matched widths are corrected and this comparison is rerun.


In [26]:
# ---- Interchange intervention: A -> B and A -> C at eps_level=2 ----
N_EVAL_TREES_IX2 = 300
EVAL_SEED_IX2 = 999
interchange_eps_results = {}

for d in EPS_D_LIST:
    for seed in SEEDS3:
        m = silent(load_shared_ckpt, eps_shared_path(2, d, seed), d, SEQ_LEN_CHECK, n_head=N_HEAD)
        eval_set_ix = build_matched_eval_set_eps(check_rules, N_EVAL_TREES_IX2, EVAL_SEED_IX2, SEQ_LEN_CHECK, eps_level=2)
        train_ids, test_ids = split_trees(N_EVAL_TREES_IX2, test_frac=0.2, seed=0)
        for tgt in ("L_B", "L_C"):
            for layer in (8, 11):
                for level in (4, 5, 6):
                    r = interchange_intervention(m, eval_set_ix, "L_A", tgt, layer, level, train_ids, test_ids)
                    interchange_eps_results[(2, d, seed, tgt, layer, level)] = r
        del m; torch.cuda.empty_cache()
        print(f"done eps=2 d={d} seed={seed}", flush=True)

print("\n=== A -> {B,C} interchange, eps_level=2 (mean over seeds) ===")
print("=== patched < baseline => A's representations don't help the target (separation) ===")
print(f"{'tgt':>4s} {'d':>4s} {'L':>2s} {'lvl':>3s} {'baseline':>9s} {'patched':>8s} {'n':>5s}")
for tgt in ("L_B", "L_C"):
    for d in EPS_D_LIST:
        for layer in (8, 11):
            for level in (4, 5, 6):
                vals = [interchange_eps_results[(2, d, s, tgt, layer, level)] for s in SEEDS3]
                base = sum(v["baseline"] for v in vals) / len(vals)
                ns = [v["n"] for v in vals if v["n"] > 0]
                patch = sum(v["patched"] for v in vals if v["n"] > 0) / max(1, len(ns))
                print(f"{tgt:>4s} {d:>4d} {layer:>2d} {level:>3d} {base:9.3f} {patch:8.3f} {sum(ns):>5d}")

done eps=2 d=8 seed=0
done eps=2 d=8 seed=7
done eps=2 d=8 seed=13
done eps=2 d=16 seed=0
done eps=2 d=16 seed=7
done eps=2 d=16 seed=13
done eps=2 d=32 seed=0
done eps=2 d=32 seed=7
done eps=2 d=32 seed=13
done eps=2 d=96 seed=0
done eps=2 d=96 seed=7
done eps=2 d=96 seed=13

=== A -> {B,C} interchange, eps_level=2 (mean over seeds) ===
=== patched < baseline => A's representations don't help the target (separation) ===
 tgt    d  L lvl  baseline  patched     n
 L_B    8  8   4     0.317    0.298  2514
 L_B    8  8   5     0.414    0.297  5787
 L_B    8  8   6     0.575    0.250 14268
 L_B    8 11   4     0.321    0.230  2514
 L_B    8 11   5     0.420    0.307  5787
 L_B    8 11   6     0.588    0.245 14268
 L_B   16  8   4     0.325    0.228  2514
 L_B   16  8   5     0.476    0.296  5787
 L_B   16  8   6     0.640    0.267 14268
 L_B   16 11   4     0.329    0.278  2514
 L_B   16 11   5     0.461    0.307  5787
 L_B   16 11   6     0.623    0.305 14268
 L_B   32  8   4     0.335   

## Outcome: why this design was abandoned

Taken together, the instruments above painted a mixed and ultimately inconclusive picture for `eps_level=1`:

- The terminal-recovery probe and CE-gap both showed a small "sharing wins" signal, but the **convergence check (above) showed the d=96 runs were still improving at 90K steps for both `eps_level=1` and `=2`** -- every d=96 number here is provisional, not converged.
- The **budget-match artifact check** showed the parameter-ratio mismatch alone could explain ~63% of the observed gap at d=32, and the direction of the d=96 crossover is also unconfirmed once that's accounted for. Only the d=16 result is mostly uncontaminated by this confound (~17% explained).
- A follow-up audit of the pre-flight ambiguity check (Section 5 / cell 38) found that restricting to contexts seen >=2 times -- the honest resolvability number -- leaves **31.5% (L_B) and 39.0% (L_C) genuinely unresolvable even at a 32-token window**, for any model regardless of capacity. The original pre-flight number (<1% still-ambiguous) was misleading because it averaged in mostly-unique, trivially-non-ambiguous contexts.
- Most importantly, a later re-read of the cost derivation found that **terminal-collapse syncretism is not actually a test of the derivation's eps** (the cost of routing a non-anchor language through an anchor-shaped shared representation space). It instead creates cost via genuine information destruction at the input -- a *dedicated*, non-shared circuit would suffer the same loss under the same lossy input. So any gap found here is evidence for source-side information loss, not anchor-bias -- a related but mechanistically distinct cost from the one the paper's theory is about.

**Decision:** abandon the lossy terminal-collapse mechanism. The next attempt (an S5-relay / non-solvable-group composition design, chosen to be genuinely non-shortcuttable rather than just locally hard) was also abandoned before training, once a simpler route was found. The project ultimately moved to a **lossless dual-role mechanism** (one terminal recoverable from tree structure, never destroyed) -- that redesign became **Experiment 7**.